In [ ]:
# %% [markdown]
# # Statistical Analysis of Credit Risk Features
# 
# This notebook demonstrates:
# - T-tests for feature comparison
# - Confidence interval calculation
# - Type I/II error trade-off analysis
# - P-value interpretation

# %% [markdown]
# ## 1. Import Libraries

# %%
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.metrics import confusion_matrix, roc_curve

# %% [markdown]
# ## 2. Load Data

# %%
df = pd.read_csv('../data/processed/credit_data_featured.csv')
print(f"Dataset shape: {df.shape}")
df.head()

# %% [markdown]
# ## 3. T-Test: Age Groups

# %%
# Split into young vs old
young = df[df['age'] < 35]['SeriousDlqin2yrs']
old = df[df['age'] >= 55]['SeriousDlqin2yrs']

# Perform t-test
t_stat, p_value = stats.ttest_ind(young, old)

print("="*50)
print("AGE GROUP T-TEST")
print("="*50)
print(f"Young (<35) default rate: {young.mean()*100:.2f}%")
print(f"Old (55+) default rate: {old.mean()*100:.2f}%")
print(f"Difference: {(young.mean() - old.mean())*100:.2f}%")
print(f"T-statistic: {t_stat:.2f}")
print(f"P-value: {p_value:.6f}")
print(f"Interpretation: {'STATISTICALLY SIGNIFICANT' if p_value < 0.05 else 'NOT significant'}")

# %% [markdown]
# ## 4. Confidence Intervals for Key Features

# %%
def calculate_ci(data, confidence=0.95):
    """Calculate confidence interval for binary data"""
    n = len(data)
    p = data.mean()
    se = np.sqrt(p * (1-p) / n)
    z = stats.norm.ppf((1 + confidence) / 2)
    return p, (p - z*se, p + z*se)

# Calculate for high utilization group
high_util = df[df['RevolvingUtilizationOfUnsecuredLines'] > 0.8]['SeriousDlqin2yrs']
low_util = df[df['RevolvingUtilizationOfUnsecuredLines'] < 0.3]['SeriousDlqin2yrs']

high_mean, high_ci = calculate_ci(high_util)
low_mean, low_ci = calculate_ci(low_util)

print("\n" + "="*50)
print("CONFIDENCE INTERVALS (95%)")
print("="*50)
print(f"High Utilization (>80%):")
print(f"  Default rate: {high_mean*100:.2f}%")
print(f"  95% CI: [{high_ci[0]*100:.2f}%, {high_ci[1]*100:.2f}%]")
print(f"\nLow Utilization (<30%):")
print(f"  Default rate: {low_mean*100:.2f}%")
print(f"  95% CI: [{low_ci[0]*100:.2f}%, {low_ci[1]*100:.2f}%]")

# %% [markdown]
# ## 5. Type I / Type II Error Trade-Off

# %%
# Assuming you have a trained model (load from saved)
import joblib

model = joblib.load('../models/saved_models/best_model.pkl')
scaler = joblib.load('../models/saved_models/scaler.pkl')
feature_cols = joblib.load('../models/saved_models/feature_columns.pkl')

# Prepare test data (you'll need your actual test set)
# This is a template - replace with your actual test data
# X_test_scaled = scaler.transform(X_test[feature_cols])
# y_proba = model.predict_proba(X_test_scaled)[:, 1]

# For demonstration, if you don't have test data loaded:
# thresholds = np.arange(0.1, 0.9, 0.05)
# type1_errors = []
# type2_errors = []
# 
# for threshold in thresholds:
#     y_pred = (y_proba >= threshold).astype(int)
#     tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
#     type1_errors.append(fp / (fp + tn))
#     type2_errors.append(fn / (fn + tp))

# Plot results (similar to the threshold analysis above)

# %% [markdown]
# ## 6. Summary Statistics Table

# %%
# Create summary table for all key features
features = ['age', 'RevolvingUtilizationOfUnsecuredLines', 
            'NumberOfTime30-59DaysPastDueNotWorse', 'MonthlyIncome']

summary_table = []

for feature in features:
    high_risk_condition = None
    low_risk_condition = None
    
    if feature == 'age':
        high_risk_condition = df[feature] < 35
        low_risk_condition = df[feature] >= 55
    elif feature == 'RevolvingUtilizationOfUnsecuredLines':
        high_risk_condition = df[feature] > 0.8
        low_risk_condition = df[feature] < 0.3
    elif feature == 'NumberOfTime30-59DaysPastDueNotWorse':
        high_risk_condition = df[feature] > 0
        low_risk_condition = df[feature] == 0
    elif feature == 'MonthlyIncome':
        high_risk_condition = df[feature] < df[feature].quantile(0.25)
        low_risk_condition = df[feature] > df[feature].quantile(0.75)
    
    if high_risk_condition is not None:
        high_rate = df[high_risk_condition]['SeriousDlqin2yrs'].mean()
        low_rate = df[low_risk_condition]['SeriousDlqin2yrs'].mean()
        diff = high_rate - low_rate
        t_stat, p_val = stats.ttest_ind(
            df[high_risk_condition]['SeriousDlqin2yrs'],
            df[low_risk_condition]['SeriousDlqin2yrs']
        )
        
        summary_table.append({
            'Feature': feature,
            'High Risk Rate': f"{high_rate*100:.2f}%",
            'Low Risk Rate': f"{low_rate*100:.2f}%",
            'Difference': f"{diff*100:.2f}%",
            'T-statistic': f"{t_stat:.1f}",
            'P-value': f"{p_val:.6f}",
            'Significant': '✅' if p_val < 0.05 else '❌'
        })

summary_df = pd.DataFrame(summary_table)
print("\n" + "="*50)
print("STATISTICAL SUMMARY TABLE")
print("="*50)
print(summary_df.to_string(index=False))

# %% [markdown]
# ## 7. Interpretation Guide

# %%
print("""
================================================================================
INTERPRETATION GUIDE FOR BUSINESS STAKEHOLDERS
================================================================================

1. P-VALUE (< 0.05)
   → "This result is statistically significant"
   → "There is less than 5% chance this happened by random luck"
   → Business meaning: We can trust this pattern is real

2. CONFIDENCE INTERVAL
   → "We are 95% confident the true value is between X and Y"
   → Business meaning: This is our honest uncertainty range

3. TYPE I ERROR (False Positive)
   → "We predicted default, but customer was good"
   → Business cost: Lost revenue, customer churn

4. TYPE II ERROR (False Negative)
   → "We predicted good, but customer defaulted"
   → Business cost: Loan loss

5. T-STATISTIC (larger = stronger evidence)
   → t > 2 = moderately strong evidence
   → t > 3 = strong evidence
   → t > 5 = very strong evidence (our delinquency t=156 is extremely strong)

================================================================================
""")